In [1]:
# !pip install reportlab
# !pip install xlsxwriter
# !pip install pyproj

In [2]:
# -------------------- Automatisierte Ordnererstellung --------------------
import os
from pathlib import Path

folders = [
    "Angepasste_Datenbanken",
    "Zwischenstände",
    "Kartierung"
]

for folder in folders:
    p = Path.cwd() / folder
    if not p.exists():
        p.mkdir(parents=True, exist_ok=True)
        print(f"Verzeichnis erstellt: {folder}")


<div class="alert alert-info">
    Im ersten Abschnitt werden Inputs und Deklarationen festgelegt, sowie In- und Output-Pfade gesetzt, um diese für spätere Verwendungszwecke erneut verwenden zu können. <br><br>
    Ebenfalls wird das anzustrebende Datenschema aufgezeigt und in einer Liste gespeichert, um dieses jederzeit abzurufen und anpassen zu können
</div>

In [3]:
import pandas as pd
import os
import numpy as np
import folium

from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [4]:
# -------------------------- Basisverzeichnis ----------------------------
base_dir = Path.cwd()

# --------------- Eingabe- und Ausgabeordner definieren ------------------
input_path = base_dir / "Gesammelte_Datenbanken"
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
data_scheme_path = base_dir / "globales_Datenschema"

print(f"Input-Pfad:  {input_path}")
print(f"Output-Pfad: {output_path}")
print(f"Schema-Pfad: {data_scheme_path}")

Input-Pfad:  F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Gesammelte_Datenbanken
Output-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition
Schema-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\globales_Datenschema


<div class="alert alert-info">
    <b>Zuerst wird eine Liste angelegt mit</b><br>
    - Zielparameter mit Einheit<br>
    - Zielparameter ohne Einheit<br>
    - Einheit des Zielparameters
</div>

In [5]:
# ----------------------- Folgende Attribute werden für die Bachelorarbeit benötigt ------------------
ATTRIBUTES = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L", 
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L", # umol
    "NH4_in_umol/L", # umol
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L", # umol
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L", # umol
    "HS_in_mmol/L",
    "Database_number",
    "Database_name"
]

# mmol = milli-Mol
# umol = Mikro-Mol


# ---------------------------------- 2. Spalte mit Entfernung aller Suffixe ------------------------------------
SUFFIXES = {
    "_in_mmol/L": "mmol/L",
    "_in_umol/L": "µmol/L",
    "_in_mV": "mV",
    "_in_m": "m",
    "_in_c": "°C",
    "_in_uS/cm": "µS/cm"
}

simplified, units = [], []

for attr in ATTRIBUTES:
    match = next(((s, u) for s, u in SUFFIXES.items() if s in attr), None)
    if match:
        s, u = match
        simplified.append(attr.replace(s, ""))
        units.append(u)
    else:
        simplified.append(attr)
        units.append("")
    
# ---------------------------------- Zusammengefügtes Array mit zwei Spalten -----------------------------------
target_structure = pd.DataFrame({
    "Original": ATTRIBUTES,
    "Simplified": simplified,
    "Unit": units
})


# --------------------------------------------- Anzeigen -------------------------------------------------------
display(target_structure)

,Original,Simplified,Unit
0,location,location,
1,well_or_spring_name,well_or_spring_name,
2,WGS84_latitude,WGS84_latitude,
3,WGS84_longitude,WGS84_longitude,
4,depth_bgl_in_m,depth_bgl,m
5,rock_type,rock_type,
6,stratigraphic_period,stratigraphic_period,
7,temperature_in_c,temperature,°C
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm
9,pH,pH,


<div class="alert alert-info">
    <b>Diese Liste wird ergänzt um</b><br>
    - eine Spalte mit Umrechnungsfaktoren von mg/L in mmol/L oder umol/L<br><br>
    Hierzu werden die Umrechnungsfaktoren
</div>

In [6]:
# ---------------------- Molmassen (g/mol) nach NIST ----------------------
MOLAR_MASS = {
    "O2": 31.9988,
    "Na": 22.989769,
    "Mg": 24.305,
    "Ca": 40.078,
    "Cl": 35.45,
    "SO4": 96.06,
    "HCO3": 61.0168,
    "Li": 6.94,
    "K": 39.0983,
    "Sr": 87.62,
    "NH4": 18.038,
    "Fe": 55.845,
    "Mn": 54.938,
    "F": 18.998,
    "NO3": 62.0049,
    "H2SiO3": 78.11,
    "HS": 33.07
}


# ---------------------- Umrechnungsfaktoren: mg/L -> mmol/L bzw. µmol/L ----------------------

# ---------------------------------- Molmassen als Spalte mappen ------------------------------
target_structure["Molar_Mass_g_per_mol"] = target_structure["Simplified"].map(MOLAR_MASS)


# --------------- Maske: nur Zeilen mit Chemie-Ziel-Einheit und bekannter Molmasse ------------
mask = target_structure["Unit"].isin(["mmol/L", "µmol/L"]) & target_structure["Molar_Mass_g_per_mol"].notna()


# -------------------- Faktor berechnen (mg/L -> mmol/L bzw. µmol/L) --------------------------
# ---------- mg/L / (g/mol) * (1000 mg / 1 g) = mmol/L
# ---------- mg/L / (g/mol) * (1_000_000 mg / 1 g) = µmol/L

target_structure["Conversion_Factor_mg_per_L_to_target"] = np.where(
    mask & target_structure["Unit"].eq("mmol/L"),
    1000 / target_structure["Molar_Mass_g_per_mol"],
    np.where(
        mask & target_structure["Unit"].eq("µmol/L"),
        1_000_000 / target_structure["Molar_Mass_g_per_mol"],
        np.nan
    )
)

# ------------------------------- Ergebnis darstellen -----------------------------------------
display(
    target_structure[["Original", "Simplified", "Unit", "Molar_Mass_g_per_mol", "Conversion_Factor_mg_per_L_to_target"]]
)

,Original,Simplified,Unit,Molar_Mass_g_per_mol,Conversion_Factor_mg_per_L_to_target
0,location,location,,NaN,NaN
1,well_or_spring_name,well_or_spring_name,,NaN,NaN
2,WGS84_latitude,WGS84_latitude,,NaN,NaN
3,WGS84_longitude,WGS84_longitude,,NaN,NaN
4,depth_bgl_in_m,depth_bgl,m,NaN,NaN
5,rock_type,rock_type,,NaN,NaN
6,stratigraphic_period,stratigraphic_period,,NaN,NaN
7,temperature_in_c,temperature,°C,NaN,NaN
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm,NaN,NaN
9,pH,pH,,NaN,NaN


<div class="alert alert-info">
    <b>Erstellen einer .pdf</b><br>
    Diese .pdf dient dazu bei händischer Auswertung der Datenbanken zu vergleichen, welche Attribute in den jeweiligen Dantesätzen vorkommen.
</div>

In [7]:
# ----------------------------- Pfad zu globalem Datenschema -----------------------------
pdf_path = Path(data_scheme_path) / "globales_Datenschema.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)

# ----------------- Angepasste Struktur für PDF (mg/L als Original) -----------------------
target_structure_pdf = target_structure.copy()
# Ersetze die Einheiten im Namen der "Original"-Spalte, damit sie den Input (mg/L) widerspiegeln
target_structure_pdf["Original"] = target_structure_pdf["Original"].str.replace("_in_mmol/L", "_in_mg/L", regex=False)
target_structure_pdf["Original"] = target_structure_pdf["Original"].str.replace("_in_umol/L", "_in_mg/L", regex=False)


# --------------------------- Daten für ReportLab vorbereiten ------------------------------
data = [target_structure_pdf.columns.tolist()] + target_structure_pdf.astype(str).values.tolist()


# ---------------------------------- .pdf erstellen ----------------------------------------
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
tbl = Table(data, repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
]))


# ---------------------------------- .pdf speichern ----------------------------------------
doc.build([tbl])
print(".pdf Pfad:", pdf_path)

.pdf Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\globales_Datenschema\globales_Datenschema.pdf


<div class="alert alert-danger">
    <h1><b>FUNKTIONEN</b></h1>
</div>

In [8]:
# -------------------------------- Hilfsfunktionen --------------------------------
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace("__+", "_", regex=True)
    )
    return df

In [9]:
# ----------------------------- Mappings, die tatsächlich im Datafram enthalten sind -----------------------------
def _available_mapping(df: pd.DataFrame, mapping: dict) -> dict:
    return {src: tgt for src, tgt in mapping.items() if src in df.columns}

In [10]:
# ---------------------------------- Einzelne Tabellenblätter aus Excel-auslesen ---------------------------------
def read_and_map(xls: pd.ExcelFile, sheet_name: str, mapping: dict, usecols=None) -> pd.DataFrame:
    if sheet_name not in xls.sheet_names:
        
        # ----------------------------- leeres DF mit 0 Zeilen
        return pd.DataFrame()
    df = pd.read_excel(xls, sheet_name=sheet_name, usecols=usecols)
    df = _normalize_columns(df)

    # --------------------------------- Fehlerbehandlung Oxygen
    if "dissolved_oxygen_mg_per_l" in df.columns and "dissolved_oxigen_mg_per_l" not in df.columns:
        df = df.rename(columns={"dissolved_oxygen_mg_per_l": "dissolved_oxigen_mg_per_l"})

    ren = _available_mapping(df, mapping)
    if not ren:
        return pd.DataFrame()

    df = df.rename(columns=ren)
    return df[list(ren.values())]

In [11]:
def csv_to_excel(csv_path: Path, excel_path: Path, sheet_name: str = "SHEET1") -> None:

    # ------------------------------ CSV einlesen ---------------------------------
    df = pd.read_csv(csv_path, low_memory=False)

    # ------------------------- In Excel schreiben --------------------------------
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        import pathlib
        _p = pathlib.Path(writer)
        if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(writer, index=False, sheet_name=sheet_name)

        # ------------------- Zugriff auf Arbeitsmappe -----------------------------
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        
        # ---------------------- Kopfzeilen-Format --------------------------------
        header_format = workbook.add_format({
            "bold": True,
            "text_wrap": True,
            "valign": "center",
            "fg_color": "#D3D3D3",
            "border": 1
        })
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_format)

        
        # ---------------------- Spaltenbreite dynamisch ---------------------------
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.set_column(i, i, min(max_len, 50))  # begrenze max. Breite

    print(f"Excel-Pfad:\n{excel_path.resolve()}")

In [12]:
# ------------------------------- mg/L zu mmol/L umrechnung ------------------------------
def mgL_to_mmolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # mg/L / (g/mol) = 1e-3 mol/L = mmol/L
    return _clean_numeric(series) / M_gmol

# ------------------------------- mg/L zu umol/L umrechnung ------------------------------
def mgL_to_umolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # (mg/L / g/mol) * 1000 = µmol/L
    return mgL_to_mmolL(series, M_gmol) * 1000.0

In [13]:
# -------------------------- Entfernen von Leerzeichen und N/D Werten ---------------------
def _clean_numeric(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"(?i)\bN/?D\b", "", regex=True)  
         .str.replace(r"[<>≈~]", "", regex=True)       
         .replace({"": np.nan})
         .pipe(pd.to_numeric, errors="coerce")
    )

<div class="alert alert-danger">
    <h1><b>DATENBANK 1</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.reflect-h2020.eu/efa/#form1 <br>
    <b>DATENBANK 1:</b> Komplette REFLECT Datenbank, stand 05.11.2025
</div>

In [14]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_reflect_path = Path.cwd() / "Zwischenstände" / "REFLECT_Datenbank"
print(f"REFLECT-Datenbank: {mid_reflect_path.resolve()}")

REFLECT-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank


In [15]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
reflect_path = input_path / "REFLECT_Horizon_2020-Data_Export_2025-11-05_20-59-36.xlsx"


# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_reflect = pd.read_excel(reflect_path)
display(df_reflect.head())

,fluid_sample_id,well_id,local_id_f,latitude_,longitude_,country,powpl_name,nuts2_name,nuts3_name,date_of_well_completion_year,...,oxigen_18_so4_thousandths,sulfur_34_so4_thousandths,sulfur_34_h2s_thousandths,carbon_13_co2_thousandths,lithium_7_thousandths,boron_11_thousandths,strontium_87sr_per_86sr_thousandths,references_for_fluid_data,remarks_fluid,Database_number
0,W_AU_001_FS_001,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
1,W_AU_001_FS_002,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
2,W_AU_001_FS_003,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
3,W_AU_001_FS_004,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Hydroisotop GmbH 347619,N/D,3
4,W_AU_001_FS_005,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Hydroisotop GmbH 347620,N/D,3


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [16]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_FLUID = "Fluid-well Data Export"
SHEET_ROCK  = "Rock-well Data Export"
SHEET_RES   = "Reservoir-well Data Export"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel ------------------------------
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive
fluid_map = {
    "nuts2_name": "location",
    "local_id_f": "well_or_spring_name",
    "latitude_": "WGS84_latitude",
    "longitude_": "WGS84_longitude",
    "well_depth_m": "depth_bgl_in_m",
    "temperature_c": "temperature_in_c",
    "electrical_conductivity_us_per_cm_ec25": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "eh_mv": "redox_potential_in_mV",
    "tds_mg_per_l": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition
    "dissolved_oxigen_mg_per_l": "O2_in_mmol/L",        
    "na_mg_per_l": "Na_in_mmol/L",
    "mg_mg_per_l": "Mg_in_mmol/L",
    "ca_mg_per_l": "Ca_in_mmol/L",
    "cl_mg_per_l": "Cl_in_mmol/L",
    "so4_mg_per_l": "SO4_in_mmol/L",
    "hco3_mg_per_l": "HCO3_in_mmol/L",
    "li_mg_per_l": "Li_in_mmol/L",
    "k_mg_per_l": "K_in_mmol/L",
    "sr_mg_per_l": "Sr_in_umol/L",
    "fe_mg_per_l": "Fe_in_mmol/L",
    "mn_mg_per_l": "Mn_in_mmol/L",
    "f_mg_per_l": "F_in_umol/L",
    "database_number": "Database_number",
}

rock_map = {
    "rock_type": "rock_type"
}

reservoir_map = {
    "age_of_formation": "stratigraphic_period"
}

In [17]:
# -------------------------------- Dateien/Ordner --------------------------------
mid_reflect_path = Path.cwd() / "Zwischenstände" / "REFLECT_Datenbank"
mid_reflect_path.mkdir(parents=True, exist_ok=True)


# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(reflect_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_fluid = read_and_map(xls, SHEET_FLUID, fluid_map)
df_rock  = read_and_map(xls, SHEET_ROCK, rock_map)
df_res   = read_and_map(xls, SHEET_RES, reservoir_map)


# -------------------------------- Zusammenführen --------------------------------
common_key = "well_or_spring_name"
can_merge = all([(common_key in df.columns) for df in [df_fluid, df_rock, df_res] if not df.empty])

if can_merge:
    df_combined = df_fluid.merge(df_rock, on=common_key, how="left").merge(df_res, on=common_key, how="left")
else:
    # --------------------------- Fallback -----------------------------------------
    df_combined = pd.concat([df_fluid, df_rock, df_res], axis=1)

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

In [18]:
# ---------------------- Rock-Type-Mapping für REFLECT ----------------------------

rock_type_mapping = {
    "Pelite": "Claystone",
    "Micritic limestone": "Limestone",
    "Psammite": "Sandstone",
    "Sparitic limestone": "Limestone",
    "Chert": "Chert",
    "Breccia": "Breccia",
    "Carbonatic siliciclastic rock": "Sandstone",
    "Gneiss": "Gneiss",
    "Granite": "Granite",
    "Dolomite": "Dolomite",
    "Andesite": "Andesite",
    "Syenite": "Salt",
    "Rhyolitic volcaniclast": "Rhyolite",
    "Andesitic volcaniclast": "Andesite",
    "Basalt": "Basalt",
    "sand and gravel": "Sand and Gravel",
    "Amphibolite": "Amphibolite",
    "Quartzite": "Quartzite",
    "Schist": "Schist",
    "Greenschist": "Schist",
    "Marble": "Marble",
    "limestone": "Limestone",
}

if "rock_type" in df_combined.columns:
    mapped_rock = df_combined["rock_type"].map(rock_type_mapping)
    df_combined["rock_type"] = mapped_rock.where(~mapped_rock.isna(), pd.NA)

In [19]:
# ----------------------------- TDS behalten ---------------------------------------
if "total_dissolved_solids_in_mmol/L" not in df_combined.columns:
    df_combined["total_dissolved_solids_in_mmol/L"] = pd.NA

if "total_dissolved_solids_in_mg/L" not in df_combined.columns and "total_dissolved_solids_in_mg/L" in fluid_map.values():
    df_combined["total_dissolved_solids_in_mg/L"] = pd.NA

In [20]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_reflect_path = mid_reflect_path / "before_conversion_reflect_mapped.csv"
import pathlib
_p = pathlib.Path(out_reflect_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_reflect_path, index=False, encoding="utf-8-sig")


print(f"REFLECT-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_reflect_path.resolve()}")

REFLECT-Mapping abgeschlossen: 5105 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [21]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_reflect_path / "before_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "before_conversion_reflect_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped_readable.xlsx


In [22]:
# Auskommentieren, wenn dies nur am Ende durchgeführt werden soll
# Einkommentieren, wenn Zwischenstände überprüft werden sollen

# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [23]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_reflect_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_reflect_path, index=False, encoding="utf-8-sig")

In [24]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [25]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}


# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan


In [26]:

# Umrechnungen (Annahme: Spalten liegen aktuell trotz Benennung mit mmol/L / umol/L in mg/L vor)
mmol_map = {
    "O2_in_mmol/L": "O2", "Na_in_mmol/L": "Na", "Mg_in_mmol/L": "Mg", "Ca_in_mmol/L": "Ca",
    "Cl_in_mmol/L": "Cl", "SO4_in_mmol/L": "SO4", "HCO3_in_mmol/L": "HCO3", "Li_in_mmol/L": "Li",
    "K_in_mmol/L": "K", "Fe_in_mmol/L": "Fe", "Mn_in_mmol/L": "Mn", "NO3_in_mmol/L": "NO3",
    "HS_in_mmol/L": "HS",
}
umol_map = {
    "Sr_in_umol/L": "Sr", "F_in_umol/L": "F", "NH4_in_umol/L": "NH4", "H2SiO3_in_umol/L": "H2SiO3",
}


# ------- mg/L -> mmol/L ---------
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L --------
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- TDS in mmol/L neu berechnen (O2 nicht inkludieren) -------------------------------------
tds_mmols_cols = [
    "Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Fe_in_mmol/L","Mn_in_mmol/L",
    "NO3_in_mmol/L","HS_in_mmol/L",
]
# Differenzierung für umol/L Spalten
tds_umols_cols = ["Sr_in_umol/L","F_in_umol/L","NH4_in_umol/L","H2SiO3_in_umol/L"]


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_reflect_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_reflect_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_reflect_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_reflect_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_reflect_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 5105 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\after_conversion_reflect_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [27]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "after_conversion_reflect_mapped_readable.xlsx"

In [28]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\after_conversion_reflect_mapped_readable.xlsx


In [29]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "after_conversion_reflect_mapped_readable.xlsx"

In [30]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
reflect_output_path = output_path / "REFLECT_cleaned_Database.csv" 
reflect_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(reflect_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(reflect_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {reflect_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\REFLECT_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [31]:
map_path = base_dir / "Kartierung"
reflect_map_path = map_path / "REFLECT_Map.html" 

In [32]:
df = df_out.copy()

# --------------------------- Nur gültige Koordinaten -------------------------------
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# -------------------------- Kartenmittelpunkt setzen -------------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------------- Punkte clustern -----------------------------------
cluster = MarkerCluster().add_to(m)

for _, r in pts.iterrows():
    folium.CircleMarker(
        location=[r["WGS84_latitude"], r["WGS84_longitude"]],
        radius=5,
        fill=True,
        fill_opacity=0.9,
        weight=1,
        popup=folium.Popup(str(r.get("well_or_spring_name", "")), max_width=300),
        tooltip=str(r.get("well_or_spring_name", "")),
    ).add_to(cluster)

# --------------------------------- Optional ----------------------------------------
folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m  # Im Notebook wird die interaktive Karte direkt angezeigt
try:
    if not reflect_map_path.parent.exists(): reflect_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(reflect_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(reflect_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 2</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.mdpi.com/2073-4441/14/1/132<br>
    <b>DATENBANK 2:</b>Hydrochemical Characteristics of Earthquake-Related Thermal Springs along the Weixi–Qiaohou Fault, Southeast Tibet Plateau , stand 06.11.2025<br>
</div>

In [33]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_mdpi_tibet_path = Path.cwd() / "Zwischenstände" / "MDPI_Tibet_Datenbank"
print(f"MDPI-Tibet-Datenbank: {mid_mdpi_tibet_path.resolve()}")

MDPI-Tibet-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank


In [34]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
mdpi_tibet_path = input_path / "MDPI_Tibet.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_mdpi_tibet = pd.read_excel(mdpi_tibet_path)

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [35]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_1 = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Sheet1 = {
    "site": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "temperature_c": "temperature_in_c",
    "conductivity_us_cm": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "na_mgl": "Na_in_mmol/L",
    "mg_mgl": "Mg_in_mmol/L",
    "ca_mgl": "Ca_in_mmol/L",
    "cl_mgl": "Cl_in_mmol/L",
    "so4_mgl": "SO4_in_mmol/L",
    "hco3_mgl": "HCO3_in_mmol/L",
    "li_ugl": "Li_in_mmol/L",
    "k_mgl": "K_in_mmol/L",
    "sr_ugl": "Sr_in_umol/L",
    "fe_ugl": "Fe_in_mmol/L",
    "mn_ugl": "Mn_in_mmol/L",
    "f_mgl": "F_in_umol/L",
    "no3_mgl": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [36]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_mdpi_tibet_path
# mdpi_tibet_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(mdpi_tibet_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, SHEET_1, Sheet1)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Metamorphic Rock als Gesteinstyp setzen</b><br>
    -> Vereinfachung in Form von Runtebrechen auf Metamorhpic Rock
</div>

In [37]:
df_combined["rock_type"] = "Metamorphic Rock"

In [38]:

# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_mdpi_tibet_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped.csv"
import pathlib
_p = pathlib.Path(out_mdpi_tibet_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_mdpi_tibet_path.resolve()}")

USGS-Mapping abgeschlossen: 52 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [39]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped.csv"
excel_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped_readable.xlsx


In [40]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung aller Werte in mg/L</b><br>
    Das manuelle Mapping hat ergeben, dass vier Parameter nicht in mg/L vorliegen:<br>
    - li_ugl (bereits umgeschrieben zu li_in_mmol/l)<br>
    - sr_ugl (bereits umgeschrieben zu sr_in_mmol/l)<br>
    - fe_ugl (bereits umgeschrieben zu fe_in_mmol/l)<br>
    - mn_ugl (bereits umgeschrieben zu mn_in_mmol/l)
</div>

In [41]:
# Diese Spalten enthalten eigentlich Werte in µg/L (falsch benannt)
ugl_cols = ["Li_in_mmol/L", "Sr_in_umol/L", "Fe_in_mmol/L", "Mn_in_mmol/L"]

for col in ugl_cols:
    if col in df_out.columns:
        # als Zahl interpretieren und von µg/L -> mg/L umrechnen
        df_out[col] = pd.to_numeric(df_out[col], errors="coerce") / 1000.0

In [42]:
# df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

In [43]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Überprüfung</b><br>
    - Es wurde manuell überprüft, dass dieser Schritt tatsächlich nur diejenigen gewünschten Spalten umrechnet<br>
    - Anschließend liegen alle Werte korrekterweise in mg/L vor<br>
    - Die Funktion kann nun angewandt werden
</div>

<div class="alert alert-info">
    <b>Schritt 2.4 | Data Insertion</b><br>
    Es ist kein Name für die "location" gegeben - dieser soll jedoch ebenfalls angefügt werden<br>
    Wir verwenden das in der Publikation beschriebene Gebiet "Weixi–Qiaohou Fault"
</div>

In [44]:
df_out["location"] = "Weixi–Qiaohou Fault"
# display(df_out["location"] )
import pathlib
_p = pathlib.Path(out_mdpi_tibet_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [45]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_mdpi_tibet_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

In [46]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [47]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [48]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_mdpi_tibet_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_mdpi_tibet_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_mdpi_tibet_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_mdpi_tibet_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_mdpi_tibet_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 52 Zeilen, 31 Spalten


Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\after_conversion_mdpi_tibet_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [49]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped.csv"
excel_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped_readable.xlsx"

In [50]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\after_conversion_mdpi_tibet_mapped_readable.xlsx


In [51]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
mdpi_tibet_output_path = output_path / "MDPI_Tibet_cleaned_Database.csv" 
mdpi_tibet_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(mdpi_tibet_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(mdpi_tibet_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {mdpi_tibet_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\MDPI_Tibet_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [52]:
map_path = base_dir / "Kartierung"
mdpi_tibet_map_path = map_path / "MDPI_Tibet_Map.html" 

In [53]:
df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen ----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# --------------------- FastMarkerCluster mit Koordinaten füllen --------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not mdpi_tibet_map_path.parent.exists(): mdpi_tibet_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(mdpi_tibet_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(mdpi_tibet_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 3</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://gdr.openei.org/submissions/425<br>
    <b>DATENBANK 3:</b> CAES 2014 Chemical Analyses of Thermal Wells and Springs in Southeastern Idaho, stand 06.11.2025<br>
</div>

In [54]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_caes2014_path = Path.cwd() / "Zwischenstände" / "CAES2024_Datenbank"
print(f"CAES2014-Tibet-Datenbank: {mid_caes2014_path.resolve()}")

CAES2014-Tibet-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank


In [55]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
caes2014_path = input_path / "CAES2014GeothermalData.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_caes2014 = pd.read_excel(caes2014_path)

# display(df_caes2014.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [56]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Data"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "county": "location",
    "samplingfeaturename": "well_or_spring_name",
    "latdegree": "WGS84_latitude",
    "longdegree": "WGS84_longitude",
    "drilleddepth_ft": "depth_bgl_in_m",
    "fluidtemperature_c": "temperature_in_c",
    "ph_field": "pH",
    "na_mgl": "Na_in_mmol/L",
    "mg_mgl": "Mg_in_mmol/L",
    "ca_mgl": "Ca_in_mmol/L",
    "cl_mgl": "Cl_in_mmol/L",
    "so4_mgl": "SO4_in_mmol/L",
    "alkalinity_mgl": "HCO3_in_mmol/L",
    "li_mgl": "Li_in_mmol/L",
    "k_mgl": "K_in_mmol/L",
    "sr_mgl": "Sr_in_umol/L",
    "f_mgl": "F_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [57]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_caes2014_path
# caes2014_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(caes2014_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()


<div class="alert alert-info">
    <b>Schritt 2.2 | Basalt als Rock Type</b><br>
    Basalt wird als Rock Type für CAES2014 angenommen
</div>

In [58]:
df_combined["rock_type"] = "Basalt"

In [59]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_caes2014_path = mid_caes2014_path / "before_conversion_caes2014_mapped.csv"
import pathlib
_p = pathlib.Path(out_caes2014_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_caes2014_path.resolve()}")

USGS-Mapping abgeschlossen: 42 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [60]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_caes2014_path / "before_conversion_caes2014_mapped.csv"
excel_path = mid_caes2014_path / "before_conversion_caes2014_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped_readable.xlsx


In [61]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von DrilledDepth_ft in Meter/L</b><br>
    - drilleddepth_ft muss in m umgewandelt werden (ist aktuell in depth_bgl_in_m gespeichert)
</div>

In [62]:
df_out["depth_bgl_in_m"] = pd.to_numeric(df_out["depth_bgl_in_m"], errors="coerce") * 0.3048

In [63]:
import pathlib
_p = pathlib.Path(out_caes2014_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [64]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_caes2014_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")

In [65]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [66]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [67]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_caes2014_path = mid_caes2014_path / "after_conversion_caes2014_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_caes2014_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_caes2014_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_caes2014_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_caes2014_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 42 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\after_conversion_caes2014_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [68]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_caes2014_path / "after_conversion_caes2014_mapped.csv"
excel_path = mid_caes2014_path / "after_conversion_caes2014_mapped_readable.xlsx"

In [69]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\after_conversion_caes2014_mapped_readable.xlsx


In [70]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
caes2014_output_path = output_path / "CAES2014_cleaned_Database.csv" 
caes2014_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(caes2014_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(caes2014_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {caes2014_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\CAES2014_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [71]:
map_path = base_dir / "Kartierung"
caes2014_map_path = map_path / "CAES2014_Map.html" 

In [72]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not caes2014_map_path.parent.exists(): caes2014_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(caes2014_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(caes2014_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 4</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://doi.pangaea.de/10.1594/PANGAEA.916088#:~:text=DGS,UNMIG%29.%20Italy.%20%28ht<br>
    <b>DATENBANK 4:</b> Alpen, stand 6.11.2025<br>
    -> Geospatial data and estimated circulation temperature, heat-flux and thermal footprint for thermal springs in the Alps [dataset]
</div>

<div class="alert alert-info">
    <b>Schritt 0 | Erstellen einer Excel</b><br>
    Da die Datei direkt in einer .csv vorliegt, wird diese zuerst in eine Excel-Datei umgewandelt, um übersichtlicher feststellen zu können, wie das Mapping ablaufen muss
</div>

In [73]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_alps_path = Path.cwd() / "Zwischenstände" / "ALPS_Datenbank"
print(f"ALPS-Datenbank: {mid_alps_path.resolve()}")

ALPS-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank


In [74]:
# ----------------------------------- Pfad einlesen -----------------------------------
alps_path = input_path / "thermal_springs_alps_with_HF_estimates.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_alps = pd.read_excel(alps_path)

# display(df_alps.head())

In [75]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_alps_path / "before_conversion_alps_mapped.csv"
excel_path = mid_alps_path / "after_first_transformation_alps.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_first_transformation_alps.xlsx


In [76]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
# csv_to_excel(csv_path, excel_path) # REMOVED PREMATURE CALL


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [77]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "SHEET1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "spring_location": "location",
    "spring_name": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "well_depth_m": "depth_bgl_in_m",
    "lithology": "rock_type",
    "temperature_degr_c": "temperature_in_c",
    "ec_s_m-1": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds_mg_l-1": "total_dissolved_solids_in_mmol/l",
    "na_mg_l-1": "Na_in_mmol/L",
    "mg_mg_l-1": "Mg_in_mmol/L",
    "ca_mg_l-1": "Ca_in_mmol/L",
    "cl_mg_l-1": "Cl_in_mmol/L",
    "so4_mg_l-1": "SO4_in_mmol/L",
    "hco3_mg_l-1": "HCO3_in_mmol/L",
    "li_mg_l-1": "Li_in_mmol/L",
    "k_mg_l-1": "K_in_mmol/L",
    "nh4_mg_l-1": "NH4_in_umol/L",
    "f_mg_l-1": "F_in_umol/L",
    "no3_mg_l-1": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

print("ALPS columns after read_and_map:", list(df_sheet1.columns))


ALPS columns after read_and_map: ['location', 'well_or_spring_name', 'WGS84_latitude', 'WGS84_longitude', 'depth_bgl_in_m', 'temperature_in_c', 'pH', 'Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L', 'Cl_in_mmol/L', 'SO4_in_mmol/L', 'HCO3_in_mmol/L', 'Li_in_mmol/L', 'K_in_mmol/L', 'Sr_in_umol/L', 'F_in_umol/L', 'Database_number']


<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [78]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_alps_path
# alps_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(alps_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

print(df_combined["rock_type"].dropna().unique())

[ 1  6  7  8  9 11  4  5  3]


<div class="alert alert-info">
    <b>Schritt 2.2 | Rock Type Mapping nach GLiM </b><br>
    -> Runterbrechen auf Zielstruktur nach REFLECT Datenbank
</div>

In [79]:
# ---------------------- Rock-Type-Mapping für REFLECT ----------------------------

rock_type_mapping = {
    1: "Sand and Gravel",
    # 2: "Sandstone",
    3: "Rhyolite",
    4: "Breccia",
    5: "Limestone",
    6: "Gypsum",
    7: "Rhyolite",
    8: "Andesite",
    9: "Basalt",
    # 10: "Granite",
    11: "Salt",
}

if "rock_type" in df_combined.columns:
    rock_codes = pd.to_numeric(df_combined["rock_type"], errors="coerce").astype("Int64")
    df_combined["rock_type"] = rock_codes.map(rock_type_mapping)
    df_combined["rock_type"] = df_combined["rock_type"].where(df_combined["rock_type"].notna(), pd.NA)

print(df_combined["rock_type"].dropna().unique())

['Sand and Gravel' 'Gypsum' 'Rhyolite' 'Andesite' 'Basalt' 'Salt'
 'Breccia' 'Limestone']


In [80]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_alps_path = mid_alps_path / "before_conversion_alps_mapped.csv"
import pathlib
_p = pathlib.Path(out_alps_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")


print(f"ALPS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_alps_path.resolve()}")

ALPS-Mapping abgeschlossen: 311 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [81]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_alps_path / "before_conversion_alps_mapped.csv"
excel_path = mid_alps_path / "before_conversion_alps_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "EC_(S_m^-1)" in uS/m</b><br>
    - drilleddepth_ft muss von S/m in uS/m umgewandelt werden (ist aktuell in "electrical_conductivity_25c_in_uS/cm" gespeichert)
</div>

In [82]:
df_out["electrical_conductivity_25c_in_uS/cm"] = pd.to_numeric(df_out["electrical_conductivity_25c_in_uS/cm"], errors="coerce") * 100

In [83]:
import pathlib
_p = pathlib.Path(out_alps_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [84]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_alps_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")

In [85]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [86]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [87]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_alps_path = mid_alps_path / "after_conversion_alps_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_alps_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_alps_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_alps_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_alps_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 311 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_conversion_alps_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [88]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_alps_path / "after_conversion_alps_mapped.csv"
excel_path = mid_alps_path / "after_conversion_alps_mapped_readable.xlsx"

In [89]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
# csv_to_excel(csv_path, excel_path)
 # REMOVED PREMATURE CALL

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

In [90]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
alps_output_path = output_path / "ALPS_cleaned_Database.csv" 
alps_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(alps_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(alps_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {alps_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\ALPS_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [91]:
map_path = base_dir / "Kartierung"
alps_map_path = map_path / "ALPS_Map.html" 

In [92]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not alps_map_path.parent.exists(): alps_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(alps_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(alps_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 5</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://open.yukon.ca/data/geothermal-thermal-springs<br>
    <b>DATENBANK 5:</b> YUKON, stand 12.11.2025<br>
    -> Geothermal Thermal Springs 
</div>

<div class="alert alert-info">
    <b>Schritt 0 | Erstellen einer Excel</b><br>
    Da die Datei direkt in einer .csv vorliegt, wird diese zuerst in eine Excel-Datei umgewandelt, um übersichtlicher feststellen zu können, wie das Mapping ablaufen muss
</div>

In [93]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_yukon_path = Path.cwd() / "Zwischenstände" / "YUKON_Datenbank"
print(f"YUKON-Datenbank: {mid_yukon_path.resolve()}")

YUKON-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank


In [94]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
yukon_path = input_path / "Yukon_Geothermal_Springs.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_yukon = pd.read_excel(yukon_path)

# display(df_yukon.head())

In [95]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_yukon_path / "before_first_transformation_yukon.csv"
excel_path = mid_yukon_path / "after_first_transformation_yukon.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_first_transformation_yukon.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_first_transformation_yukon.xlsx


In [96]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
# csv_to_excel(csv_path, excel_path) # REMOVED PREMATURE CALL (Auto-Fix)


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [97]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "SHEET1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location": "location",
    "name": "well_or_spring_name",
    "latitude_dd": "WGS84_latitude",
    "longitude_dd": "WGS84_longitude",
    "conductivity": "electrical_conductivity_25c_in_uS/cm",
    "water_temp": "temperature_in_c",
    "ph": "pH",
    "na_mg_l": "Na_in_mmol/L",
    "mg_mg_l": "Mg_in_mmol/L",
    "ca_mg_l": "Ca_in_mmol/L",
    "cl_mg_l": "Cl_in_mmol/L",
    "so4_mg_l": "SO4_in_mmol/L",
    "hco3_mg_l": "HCO3_in_mmol/L",
    "k_mg_l": "K_in_mmol/L",
    "f_mg_l": "F_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [98]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_yukon_path
# yukon_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(yukon_path)

# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Granite</b><br>
    Granite ist das überwiegende Vorkommen im Yukon
</div>

In [99]:
df_combined["rock_type"] = "Granite"

In [100]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_yukon_path = mid_yukon_path / "before_conversion_yukon_mapped.csv"
import pathlib
_p = pathlib.Path(out_yukon_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_yukon_path, index=False, encoding="utf-8-sig")


print(f"YUKON-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_yukon_path.resolve()}")

YUKON-Mapping abgeschlossen: 88 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [101]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_yukon_path / "before_conversion_yukon_mapped.csv"
excel_path = mid_yukon_path / "before_conversion_yukon_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [102]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_yukon_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_yukon_path, index=False, encoding="utf-8-sig")

In [103]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [104]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [105]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_yukon_path = mid_yukon_path / "after_conversion_yukon_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_yukon_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_yukon_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_yukon_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_yukon_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 88 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_conversion_yukon_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [106]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_yukon_path / "after_conversion_yukon_mapped.csv"
excel_path = mid_yukon_path / "after_conversion_yukon_mapped_readable.xlsx"

In [107]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_conversion_yukon_mapped_readable.xlsx


In [108]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
yukon_output_path = output_path / "YUKON_cleaned_Database.csv" 
yukon_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(yukon_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(yukon_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {yukon_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\YUKON_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [109]:
map_path = base_dir / "Kartierung"
yukon_map_path = map_path / "YUKON_Map.html" 

In [110]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not yukon_map_path.parent.exists(): yukon_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(yukon_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(yukon_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 6</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://repositorio.uchile.cl/bitstream/handle/2250/149568/Geochemistry-of-thermal-waters.pdf<br>
    <b>DATENBANK 6:</b> CHILE Datenbank, stand 13.11.2025, Geochemistry of thermal waters in the Southern Volcanic Zone, Chile –
Implications for structural controls on geothermal fluid composition
</div>

In [111]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_chile_path = Path.cwd() / "Zwischenstände" / "SVZ-CHILE_Datenbank"
print(f"CHILE-Datenbank: {mid_chile_path.resolve()}")

CHILE-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank


In [112]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
chile_path = input_path / "SVZ_Chile_Table1_merged.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_chile = pd.read_excel(chile_path)

# display(df_chile.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [113]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "TABLE1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location_name": "location",
    "fault_domain": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "t_c": "temperature_in_c",
    "ph": "pH",
    "tds_ppm": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "na_ppm": "Na_in_mmol/L",
    "mg_ppm": "Mg_in_mmol/L",
    "ca_ppm": "Ca_in_mmol/L",
    "cl_ppm": "Cl_in_mmol/L",
    "so4_ppm": "SO4_in_mmol/L",
    "hco3_ppm": "HCO3_in_mmol/L",
    "li_ppm": "Li_in_mmol/L",
    "k_ppm": "K_in_mmol/L",
    "sr_ppb": "Sr_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [114]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_chile_path
# chile_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(chile_path)



# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# display(df_sheet1.head())
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Basalt als Rock Type für Chile Database</b><br>
    -> Vereinfachung auf Basalt
</div>

In [115]:
df_combined["rock_type"] = "Andesite"

In [116]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()

display(df_out.head())
# ------------------------------ Export ---------------------------------------------
out_chile_path = mid_chile_path / "before_conversion_chile_mapped.csv"
import pathlib
_p = pathlib.Path(out_chile_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")


print(f"CHILE-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_chile_path.resolve()}")

,location,well_or_spring_name,WGS84_latitude,WGS84_longitude,depth_bgl_in_m,rock_type,stratigraphic_period,temperature_in_c,electrical_conductivity_25c_in_uS/cm,pH,...,Sr_in_umol/L,NH4_in_umol/L,Fe_in_mmol/L,Mn_in_mmol/L,F_in_umol/L,NO3_in_mmol/L,H2SiO3_in_umol/L,HS_in_mmol/L,Database_number,Database_name
0,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,Andesite,<NA>,68.0,<NA>,3.9,...,78.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13,<NA>
1,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,Andesite,<NA>,82.0,<NA>,2.6,...,47.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13,<NA>
2,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,Andesite,<NA>,91.0,<NA>,2.4,...,143.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13,<NA>
3,Trapa Trapa,LOFS,-37.82667,-71.13000,<NA>,Andesite,<NA>,45.0,<NA>,7.8,...,305.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13,<NA>
4,Copahue Vn.,LOFS,-37.85000,-71.17000,<NA>,Andesite,<NA>,87.0,<NA>,2.7,...,141.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13,<NA>


CHILE-Mapping abgeschlossen: 30 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [117]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_chile_path / "before_conversion_chile_mapped.csv"
excel_path = mid_chile_path / "before_conversion_chile_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "Li", welches aktuell in umol/L vorliegt zu mmol/L</b><br>
    - ist aktuell in "Li_in_mmol/L" gespeichert
</div>

In [118]:
df_out["Li_in_mmol/L"] = pd.to_numeric(df_out["Li_in_mmol/L"], errors="coerce") / 1000

In [119]:
import pathlib
_p = pathlib.Path(out_chile_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [120]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_chile_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")

In [121]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [122]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [123]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_chile_path = mid_chile_path / "after_conversion_chile_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_chile_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_chile_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_chile_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_chile_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 30 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\after_conversion_chile_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [124]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_chile_path / "after_conversion_chile_mapped.csv"
excel_path = mid_chile_path / "after_conversion_chile_mapped_readable.xlsx"

In [125]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\after_conversion_chile_mapped_readable.xlsx


In [126]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
chile_output_path = output_path / "SVZ-CHILE_cleaned_Database.csv" 
chile_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(chile_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(chile_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {chile_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\SVZ-CHILE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [127]:
map_path = base_dir / "Kartierung"
chile_map_path = map_path / "SVZ-Chile_Map.html" 

In [128]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not chile_map_path.parent.exists(): chile_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(chile_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(chile_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 7</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.cell.com/heliyon/fulltext/S2405-8440(23)04561-9?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS2405844023045619%3Fshowall%3Dtrue<br>
    <b>DATENBANK 7:</b> GANDAKI Datenbank, stand 10.11.2025, Gandaki Province, Nepal 
</div>

In [129]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_gandaki_path = Path.cwd() / "Zwischenstände" / "GANDAKI_Datenbank"
print(f"GANDAKI-Datenbank: {mid_gandaki_path.resolve()}")

GANDAKI-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank


In [130]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
gandaki_path = input_path / "mmc1_updated.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_gandaki = pd.read_excel(gandaki_path)

# display(df_gandaki.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [131]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location": "location",
    "id": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "temp_c": "temperature_in_c",
    "ec_us_cm": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds_mg_l": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "na_mg_l": "Na_in_mmol/L",
    "mg_mg_l": "Mg_in_mmol/L",
    "ca_mg_l": "Ca_in_mmol/L",
    "cl_mg_l": "Cl_in_mmol/L",
    "so4_mg_l": "SO4_in_mmol/L",
    "hco3_mg_l": "HCO3_in_mmol/L",
    "k_mg_l": "K_in_mmol/L",
    "fe_mg_l": "Fe_in_mmol/L",
    "f_mg_l": "F_in_umol/L",
    "no3_mg_l": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [132]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_gandaki_path
# gandaki_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(gandaki_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Metamorphic Rock</b><br>
    Gebiet ist Greenschistlastig (und generell Schieferbetont) -> Metamorphe Gesteine
</div>

In [133]:
df_combined["rock_type"] = "Metamorphic Rock"

In [134]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_gandaki_path = mid_gandaki_path / "before_conversion_gandaki_mapped.csv"
import pathlib
_p = pathlib.Path(out_gandaki_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_gandaki_path, index=False, encoding="utf-8-sig")


print(f"GANDAKI-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_gandaki_path.resolve()}")

GANDAKI-Mapping abgeschlossen: 12 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [135]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_gandaki_path / "before_conversion_gandaki_mapped.csv"
excel_path = mid_gandaki_path / "before_conversion_gandaki_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [136]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_gandaki_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_gandaki_path, index=False, encoding="utf-8-sig")

In [137]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [138]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [139]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_gandaki_path = mid_gandaki_path / "after_conversion_gandaki_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_gandaki_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_gandaki_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_gandaki_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_gandaki_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 12 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\after_conversion_gandaki_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [140]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_gandaki_path / "after_conversion_gandaki_mapped.csv"
excel_path = mid_gandaki_path / "after_conversion_gandaki_mapped_readable.xlsx"

In [141]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\after_conversion_gandaki_mapped_readable.xlsx

In [142]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
gandaki_output_path = output_path / "GANDAKI_cleaned_Database.csv" 
gandaki_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(gandaki_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(gandaki_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {gandaki_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\GANDAKI_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [143]:
map_path = base_dir / "Kartierung"
gandaki_map_path = map_path / "GANDAKI_Map.html" 

In [144]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not gandaki_map_path.parent.exists(): gandaki_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(gandaki_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(gandaki_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 8</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://essd.copernicus.org/articles/13/4847/2021/<br>
    <b>DATENBANK 8:</b> HESSEN Datenbank, stand 10.11.2025, HESSEN_Datenbank
</div>

In [145]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_hessen_path = Path.cwd() / "Zwischenstände" / "HESSEN_Datenbank"
print(f"HESSEN-Datenbank: {mid_hessen_path.resolve()}")

HESSEN-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank


In [146]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
hessen_path = input_path / "Hydrochemical_Database_Hessen_3D_2-0_RSc_202109.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_hessen = pd.read_excel(hessen_path)

# display(df_hessen.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [147]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "town": "location",
    "well_or_spring_name": "well_or_spring_name",
    "coordinates_r": "WGS84_latitude",
    "coordinates_h": "WGS84_longitude",
    "final_depth": "depth_bgl_in_m",
    "rock_type": "rock_type",
    "period": "stratigraphic_period",
    "temperature": "temperature_in_c",
    "electric": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "redox": "redox_potential_in_mV",
    "total_dissolved_solids_calculated": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "oxigen": "O2_in_mmol/L",
    "sodium": "Na_in_mmol/L",
    "magnesium": "Mg_in_mmol/L",
    "calcium": "Ca_in_mmol/L",
    "chloride": "Cl_in_mmol/L",
    "sulphate": "SO4_in_mmol/L",
    "bicarbonate": "HCO3_in_mmol/L",
    "lithium": "Li_in_mmol/L",
    "potassium": "K_in_mmol/L",
    "strontium": "Sr_in_umol/L",
    "ammonium": "NH4_in_umol/L",
    "iron": "Fe_in_mmol/L",
    "manganese": "Mn_in_mmol/L",
    "fluoride": "F_in_umol/L",
    "nitrate": "NO3_in_mmol/L",
    "silica": "H2SiO3_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [148]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_hessen_path
# hessen_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(hessen_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()


<div class="alert alert-info">
    <b>Schritt 2.2 | Hessen Datenbank</b><br>
    Rock-Type Mapping etwas komplizierter:<br>
    - Zeilen mit bspw. "Sandstone, Limestone" werden in zwei identische Zeilen aufgesplitted und einmal mit "Sandstone" und einmal "Limestone" besetzt<br>
    - Unnötige Gesteinstypen werden vernachlässigt
</div>

In [149]:
# Komplexeres Mappiung für Hessen Datenbank
hessen_rock_mapping = {
    "quartzite": ["Quartzite"],
    "sandstone": ["Sandstone"],
    "unconsolidated rocks": ["Sand and Gravel"],
    "arkose, sandstone, conglomerate": ["Sandstone"],
    "pelite": ["Claystone"],
    "slate, sandstone": ["Slate", "Sandstone"],
    "dolomite": ["Dolomite"],
    "greenschist": ["Schist"],
    "gneiss, greenschist": ["Gneiss", "Greenschist"],
    "pelite, halite rock": ["Claystone", "Salt"],
    "sandstone, limestone, granite": ["Sandstone", "Limestone", "Granite"],
    "unconsolidated rocks, carbonate bearing silt": ["Sand and Gravel"],
    "limestone": ["Limestone"],
    "slate": ["Slate"],
    "phyllite": ["Phyllite"],
    "marl": ["Marl"],
    "pelite, sandstone, conglomerate": ["Sandstone", "Claystone"],
    "chert": ["Chert"],
    "graywacke, dolerite": ["Dolerite"],
    "gypsum": ["Gypsum"],
    "dolomite, pelite": ["Dolomite", "Claystone"],
    "graywacke, slate": ["Slate"],
    "unconsolidated rocks, marl": ["Marl", "Sand and Gravel"],
    "quartzite, slate": ["Quartzite", "Slate"],
    "pelite, sandstone": ["Claystone", "Sandstone"],
    "slate, pelite": ["Slate", "Claystone"],
    "dolerite": ["Dolerite"],
    "tuff, hawaiite": ["Tuff"],
    "muscovite‑biotite‑gneiss": ["Gneiss"],
    "dolomite, pelite, limestone, anhydrite": ["Dolomite", "Claystone", "Limestone"],
    "marl, pelite, limestone": ["Marl", "Claystone", "Limestone"],
    "arkose": ["Sandstone"],
    "limestone, arkose, pelite": ["Limestone", "Sandstone", "Claystone"],
    "slate, volcanic rocks": ["Slate"],
    "slate, graywacke": ["Slate", "Graywacke"],
    "marl, dolomite": ["Marl", "Dolomite"],
    "slate, graywacke, dolerite": ["Slate", "Graywacke", "Dolerite"],
    "gneiss, phyllite": ["Gneiss", "Phyllite"],
}

if "rock_type" in df_combined.columns:
    expanded_rows = []
    for _, row in df_combined.iterrows():
        original_value = str(row["rock_type"]).strip()
        
        # -------------- bei fehlendem Eintrag oder leerem String NaN
        if pd.isna(row["rock_type"]) or original_value == "":
            new_row = row.copy()
            new_row["rock_type"] = pd.NA
            expanded_rows.append(new_row)
            continue

        # -------------- Kleinschreibung für das Mapping
        key = original_value.lower()
        if key in hessen_rock_mapping:
            
            # ------------ für jeden gemappten Gesteinstyp eine separate Zeile erzeugen
            for mapped_type in hessen_rock_mapping[key]:
                new_row = row.copy()
                new_row["rock_type"] = mapped_type
                expanded_rows.append(new_row)
        else:
            # ---------- unbekannte Einträge uf NaN setzen und Zeile behalten
            new_row = row.copy()
            new_row["rock_type"] = pd.NA
            expanded_rows.append(new_row)

    df_combined = pd.DataFrame(expanded_rows)

In [150]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_hessen_path = mid_hessen_path / "before_conversion_hessen_mapped.csv"
import pathlib
_p = pathlib.Path(out_hessen_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_hessen_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_hessen_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 1144 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [151]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_hessen_path / "before_conversion_hessen_mapped.csv"
excel_path = mid_hessen_path / "before_conversion_hessen_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [152]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_hessen_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_hessen_path, index=False, encoding="utf-8-sig")

In [153]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [154]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [155]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_hessen_path = mid_hessen_path / "after_conversion_hessen_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_hessen_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_hessen_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_hessen_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_hessen_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 1144 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\after_conversion_hessen_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [156]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_hessen_path / "after_conversion_hessen_mapped.csv"
excel_path = mid_hessen_path / "after_conversion_hessen_mapped_readable.xlsx"

In [157]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\after_conversion_hessen_mapped_readable.xlsx


In [158]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
hessen_output_path = output_path / "HESSEN_cleaned_Database.csv" 
hessen_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(hessen_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(hessen_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {hessen_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\HESSEN_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [159]:
map_path = base_dir / "Kartierung"
hessen_map_path = map_path / "HESSEN_Map.html" 

In [160]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not hessen_map_path.parent.exists(): hessen_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(hessen_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(hessen_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 9</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.mdpi.com/2073-4441/14/4/550#:~:text=The%2043%20hot%20springs%20were,rate%20of%20slip%20in%20the<br>
    <b>DATENBANK 9:</b> MDPI Xianshuihe Datenbank, stand 12.11.2025
</div>

In [161]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_xian_path = Path.cwd() / "Zwischenstände" / "MDPI-Xianshuihe_Datenbank"
print(f"XIANSHUIHE-Datenbank: {mid_xian_path.resolve()}")

XIANSHUIHE-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank


In [162]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
xian_path = input_path / "MPDI_Xianshuihe.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_xian = pd.read_excel(xian_path)

# display(df_xian.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [163]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "no": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "temperature": "temperature_in_c",
    "ec": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "k": "K_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [164]:
# -------------------------------- Dateien/Ordner --------------------------------
# mix_xian_path
# xian_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(xian_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Sandstone als allgemeiner Rock Type für Xianshuie Datenbank</b><br>
    Überwiegender Gesteinstyp insgesamt Metamorphic Rock (auch auftreten von metamorphem Sandstein, ...). Granit kommt vor, aber eher lokal auf einzelne Stellen begrenzt. Wahl von Metamorphic Rock
</div>

In [165]:
df_combined["rock_type"] = "Metamorphic Rock"

In [166]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_xian_path = mid_xian_path / "before_conversion_xian_mapped.csv"
import pathlib
_p = pathlib.Path(out_xian_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_xian_path, index=False, encoding="utf-8-sig")


print(f"XIAN-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_xian_path.resolve()}")

XIAN-Mapping abgeschlossen: 280 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [167]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_xian_path / "before_conversion_xian_mapped.csv"
excel_path = mid_xian_path / "before_conversion_xian_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [168]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_xian_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_xian_path, index=False, encoding="utf-8-sig")

In [169]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [170]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [171]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_xian_path = mid_xian_path / "after_conversion_xian_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_xian_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_xian_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_xian_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_xian_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 280 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\after_conversion_xian_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [172]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_xian_path / "after_conversion_xian_mapped.csv"
excel_path = mid_xian_path / "after_conversion_xian_mapped_readable.xlsx"

In [173]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\after_conversion_xian_mapped_readable.xlsx


In [174]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
xian_output_path = output_path / "XIANSHUIHE_cleaned_Database.csv" 
xian_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(xian_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(xian_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {xian_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\XIANSHUIHE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [175]:
map_path = base_dir / "Kartierung"
xian_map_path = map_path / "XIANSHUIHE_Map.html" 

In [176]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not xian_map_path.parent.exists(): xian_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(xian_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(xian_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 10</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.nature.com/articles/s41598-021-03701-1?error=cookies_not_supported&code=d245a11e-6034-45f3-b6f9-c53fc736651e#article-info<br>
    <b>DATENBANK 10:</b> Australia Mataranka, stand 12.11.2025
</div>

In [177]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_australia_path = Path.cwd() / "Zwischenstände" / "AUSTRALIA-MATARANKA_Datenbank"
print(f"AUSTRALIA-Datenbank: {mid_australia_path.resolve()}")

AUSTRALIA-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank


In [178]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
australia_path = input_path / "Australia_Mataranka.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_australia = pd.read_excel(australia_path)

# display(df_australia.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [179]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "cla_flow_path": "location",
    "bore_spring": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "na_mg-l": "Na_in_mmol/L",
    "mg_mg-l": "Mg_in_mmol/L",
    "ca_mg-l": "Ca_in_mmol/L",
    "cl_mg-l": "Cl_in_mmol/L",
    "so4_mg-l": "SO4_in_mmol/L",
    "alkalinity_meq-l": "HCO3_in_mmol/L",
    "k_mg-l": "K_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [180]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_australia_path
# australia_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(australia_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Limestone als allgemeiner Rock Type für Australia Datenbank</b><br>
    CLA Formation geht hauptsächlich aus Limestone (daher Name LA -> Limestone Quifer) vor
</div>

In [181]:
df_combined["rock_type"] = "Limestone"

In [182]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_australia_path = mid_australia_path / "before_conversion_australia_mapped.csv"
import pathlib
_p = pathlib.Path(out_australia_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_australia_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 84 Zeilen, 31 Spalten


CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [183]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_australia_path / "before_conversion_australia_mapped.csv"
excel_path = mid_australia_path / "before_conversion_australia_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "alkalinity_meq-l" in mg/L</b><br>
    - ist aktuell in "HCO3_in_mmol/L" gespeichert | Faktor 61.0168
</div>

In [184]:
df_out["HCO3_in_mmol/L"] = pd.to_numeric(df_out["HCO3_in_mmol/L"], errors="coerce") * 61.0168

In [185]:
import pathlib
_p = pathlib.Path(out_australia_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [186]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_australia_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")

In [187]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [188]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [189]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_australia_path = mid_australia_path / "after_conversion_australia_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_australia_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_australia_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_australia_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_australia_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 84 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\after_conversion_australia_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [190]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_australia_path / "after_conversion_australia_mapped.csv"
excel_path = mid_australia_path / "after_conversion_australia_mapped_readable.xlsx"

In [191]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\after_conversion_australia_mapped_readable.xlsx


In [192]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
australia_output_path = output_path / "AUSTRALIA-MATARANKA_cleaned_Database.csv" 
australia_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(australia_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(australia_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {australia_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\AUSTRALIA-MATARANKA_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [193]:
map_path = base_dir / "Kartierung"
australia_map_path = map_path / "AUSTRALIA-MATARANKA_Map.html" 

In [194]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not australia_map_path.parent.exists(): australia_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(australia_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(australia_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 11</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.tshe.org/ea/pdf/EA14(3)/EA14(3)_04.pdf#:~:text=Geochemical%20data%20from%20more%20than,The%20geochemical%20data<br>
    <b>DATENBANK 11:</b> THAILAND Datenbank, stand 12.11.2025
</div>

In [195]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_thailand_path = Path.cwd() / "Zwischenstände" / "THAILAND_Datenbank"
print(f"THAILAND-Datenbank: {mid_thailand_path.resolve()}")

THAILAND-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank


In [196]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
thailand_path = input_path / "Southern_Thailand_Parsed.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_thailand = pd.read_excel(thailand_path)

# display(df_thailand.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [197]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "geothermal_province": "location",
    "hot_spring": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "ph": "pH",
    "surface_temperature": "temperature_in_c",
    "na_mg-l": "Na_in_mmol/L",
    "mg_mg-l": "Mg_in_mmol/L",
    "ca_mg-l": "Ca_in_mmol/L",
    "cl_mg-l": "Cl_in_mmol/L",
    "so4_mg-l": "SO4_in_mmol/L",
    "hco3_mg-l": "HCO3_in_mmol/L",
    "k_mg-l": "K_in_mmol/L",
    "fe_mg-l": "Fe_in_mmol/L",
    "mn_mg-l": "Mn_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [198]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_thailand_path
# thailand_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(thailand_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Anpassen auf neue Lithologie</b><br>
    Bei über 60°C wird Granite gesetzt, bei unter 60° wird Sandstone gesetzt
</div>

In [199]:
# ---------- Temperatur zur Referenz verwenden
if "temperature_in_c" in df_combined.columns:
    temps = pd.to_numeric(df_combined["temperature_in_c"], errors="coerce")

    df_combined["rock_type"] = pd.NA

    # ------ Größer als 60 wird zu Granite
    df_combined.loc[temps > 60, "rock_type"] = "Granite"

    # ------ Kleiuner gleich 60 wird zu Sandstone
    df_combined.loc[temps <= 60, "rock_type"] = "Sandstone"

In [200]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_thailand_path = mid_thailand_path / "before_conversion_thailand_mapped.csv"
import pathlib
_p = pathlib.Path(out_thailand_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_thailand_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_thailand_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 31 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.3 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [201]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_thailand_path / "before_conversion_thailand_mapped.csv"
excel_path = mid_thailand_path / "before_conversion_thailand_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [202]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_thailand_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_thailand_path, index=False, encoding="utf-8-sig")

C:\Users\lucca\AppData\Local\Temp\ipykernel_20560\356138064.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)


In [203]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [204]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [205]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_thailand_path = mid_thailand_path / "after_conversion_thailand_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_thailand_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_thailand_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_thailand_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_thailand_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 31 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\after_conversion_thailand_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [206]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_thailand_path / "after_conversion_thailand_mapped.csv"
excel_path = mid_thailand_path / "after_conversion_thailand_mapped_readable.xlsx"

In [207]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\after_conversion_thailand_mapped_readable.xlsx


In [208]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
thailand_output_path = output_path / "THAILAND_cleaned_Database.csv" 
thailand_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(thailand_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(thailand_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {thailand_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\THAILAND_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [209]:
map_path = base_dir / "Kartierung"
thailand_map_path = map_path / "THAILAND_Map.html" 

In [210]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not thailand_map_path.parent.exists(): thailand_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(thailand_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(thailand_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK VORLETZTE</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://catalog.data.gov/dataset/argonne-geothermal-geochemical-database-v2-0-53978/resource/35bedd29-61f0-418e-8233-0511640a8d9e#:~:text=Argonne%20Geothermal%20Geochemical%20Database%20v2_00<br>
    <b>DATENBANK VORLETZTE:</b> ARGONNE Datenbank, stand 12.11.2025, Argonne Geothermal Geochemical Database v2_00.xlsx
</div>

In [211]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_argonne_path = Path.cwd() / "Zwischenstände" / "ARGONNE_Datenbank"
print(f"ARGONNE-Datenbank: {mid_argonne_path.resolve()}")

ARGONNE-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank


In [212]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
argonne_path = input_path / "Argonne_Geothermal_Geochemical_Database_v2_00.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_argonne = pd.read_excel(argonne_path)

# display(df_argonne.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [213]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "ALLDATA"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "state": "location",
    "name": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "depth_ft": "depth_bgl_in_m",
    "temperature_c": "temperature_in_c",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "li": "Li_in_mmol/L",
    "k": "K_in_mmol/L",
    "sr": "Sr_in_umol/L",
    "nh4_n": "NH4_in_umol/L",
    "fe": "Fe_in_mmol/L",
    "mn": "Mn_in_mmol/L",
    "f": "F_in_umol/L",
    "no3_n": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [214]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_argonne_path
# argonne_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(argonne_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_argonne_path = mid_argonne_path / "before_conversion_argonne_mapped.csv"
import pathlib
_p = pathlib.Path(out_argonne_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")


print(f"ARGONNE-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_argonne_path.resolve()}")

ARGONNE-Mapping abgeschlossen: 51078 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [215]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_argonne_path / "before_conversion_argonne_mapped.csv"
excel_path = mid_argonne_path / "before_conversion_argonne_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "depth_ft" in m</b><br>
    - drilleddepth_ft muss von ft in m umgewandelt werden (ist aktuell in "depth_bgl_in_m" gespeichert)
</div>

In [216]:
df_out["depth_bgl_in_m"] = pd.to_numeric(df_out["depth_bgl_in_m"], errors="coerce") * 0.3048

In [217]:
import pathlib
_p = pathlib.Path(out_argonne_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [218]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_argonne_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")

In [219]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [220]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path, low_memory=False)


# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [221]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_argonne_path = mid_argonne_path / "after_conversion_argonne_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_argonne_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_argonne_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_argonne_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_argonne_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 51078 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\after_conversion_argonne_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [222]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_argonne_path / "after_conversion_argonne_mapped.csv"
excel_path = mid_argonne_path / "after_conversion_argonne_mapped_readable.xlsx"

In [223]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\after_conversion_argonne_mapped_readable.xlsx


In [224]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
argonne_output_path = output_path / "ARGONNE_cleaned_Database.csv" 
argonne_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(argonne_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(argonne_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {argonne_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\ARGONNE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [225]:
map_path = base_dir / "Kartierung"
argonne_map_path = map_path / "Argonne_Map.html" 

In [226]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not argonne_map_path.parent.exists(): argonne_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(argonne_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(argonne_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK FINAL</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.sciencebase.gov/catalog/item/59d25d63e4b05fe04cc235f9<br>
    <b>DATENBANK final:</b> Komplette USGS Datenbank, stand 07.11.2025<br>
    -> Kommt an letzter Stelle, da lange Ladezeiten (auf Grund von über 100.000 Datenzeilen)
</div>

In [227]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_usgs_path = Path.cwd() / "Zwischenstände" / "USGS_Datenbank"
print(f"USGS-Datenbank: {mid_usgs_path.resolve()}")

USGS-Datenbank: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank


In [228]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
usgs_path = input_path / "USGSPWDBv2.3n.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_usgs = pd.read_excel(usgs_path)

# display(df_usgs.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [229]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_1 = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
    "Database_name",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive
Sheet1 = {
    "county": "location",
    "wellname": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "depthlower": "depth_bgl_in_m",
    "lithology": "rock_type",
    "period": "stratigraphic_period",
    "temp": "temperature_in_c",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durch Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "li": "Li_in_mmol/L",
    "k": "K_in_mmol/L",
    "sr": "Sr_in_umol/L",
    "nh4": "NH4_in_umol/L",
    "fetot": "Fe_in_mmol/L",
    "mn": "Mn_in_mmol/L",
    "f": "F_in_umol/L",
    "no3": "NO3_in_mmol/L",
    "hs": "HS_in_mmol/L",
    "database_number": "Database_number",
    "lithology": "temporary_lithology",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [230]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_usgs_path
# usgs_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(usgs_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, SHEET_1, Sheet1)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()

<div class="alert alert-info">
    <b>Schritt 2.2 | Auswertung vorhandener Rock Types</b><br>
    Auswertung der bereits vorhandenen Lithologie
</div>

In [231]:
# Einzigartige Lithology-Werte + Häufigkeiten ausgeben
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

if "temporary_lithology" in df_combined.columns:
    lithology_counts = (
        df_combined["temporary_lithology"]
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
    )
    print(lithology_counts)
else:
    print("Spalte 'Lithology' nicht gefunden. Vorhandene Spalten:", list(df_combined.columns))

temporary_lithology
nan                                                 86006
Sandstone                                           15092
Limestone                                            4332
Coal                                                 3450
Dolomite                                             2480
Dolomite, Limestone                                   498
Shale                                                 428
Limestone, Sandstone                                  268
Unknown                                               257
Carbonate                                             255
Limestone, Shale                                      191
Ss                                                    181
Dolomite, Sandstone                                   181
Sandstone, Shale                                      158
Limestone; Dolostone                                  143
Anhydrite, Sandstone                                  134
Dolomite, Limestone, Sandstone                      

<div class="alert alert-info">
    <b>Schritt 2.3 | Rock Type Mapping</b><br>
    Nach Auswertung Durchführung von Mapping - erstmal nur mit bereits verfügbaren Daten
</div>

In [232]:
import re

usgs_rock_mapping = {
    # ------------- Mapping gemäß Excel-Datei (funktionsweise wie bei Hessen Datenbank)
    "sandstone": ["Sandstone"],
    "limestone": ["Limestone"],
    "coal": ["Coal"],
    "dolomite": ["Dolomite"],
    "dolomite, limestone": ["Dolomite", "Limestone"],
    "shale": ["Shale"],
    "limestone, sandstone": ["Limestone", "Sandstone"],
    "carbonate": ["Limestone"],
    "limestone, shale": ["Limestone", "Shale"],
    "ss": ["Sandstone"],
    "dolomite, sandstone": ["Dolomite", "Sandstone"],
    "sandstone, shale": ["Sandstone", "Shale"],
    "limestone, dolostone": ["Limestone"],
    "anhydrite, sandstone": ["Sandstone", "Anhydrite"],
    "dolomite, limestone, sandstone": ["Dolomite", "Limestone", "Sandstone"],
    "anhydrite, dolomite": ["Dolomite"],
    "chert": ["Chert"],
    "shale, black": ["Shale"],
    "sand": ["Sand and Gravel"],
    "sandstone, siltstone": ["Sandstone", "Siltstone"],
    "sandstone, shale, siltstone": ["Sandstone", "Shale", "Siltstone"],
    "anhydrite": ["Anhydrite"],
    "dolomite, shale": ["Dolomite", "Shale"],
    "chert, dolomite, sandstone": ["Chert", "Dolomite", "Sandstone"],
    "anhydrite, dolomite, sandstone": ["Anhydrite", "Dolomite", "Sandstone"],
    "dolomite, sandstone, shale": ["Dolomite", "Sandstone", "Shale"],
    "sandstone, shale, other": ["Sandstone", "Shale"],
    "sandstone, other": ["Sandstone"],
    "chert, dolomite": ["Chert", "Dolomite"],
    "siltstone": ["Siltstone"],
    "chert, dolomite, limestone": ["Chert", "Dolomite", "Limestone"],
    "anhydrite, dolomite, sandstone, shale": ["Anhydrite", "Dolomite", "Sandstone", "Shale"],
    "ss, dolomitic?": ["Sandstone", "Dolomite"],
    "chalk": ["Limestone"],
    "anhydrite, dolomite, shale": ["Anhydrite", "Dolomite", "Shale"],
    "chert, limestone": ["Chert", "Limestone"],
    "anhydrite, dolomite, limestone": ["Anhydrite", "Dolomite", "Limestone"],
    "limestone, sandstone, siltstone": ["Limestone", "Sandstone", "Siltstone"],
    "chert, sandstone": ["Chert", "Sandstone"],
    "limestone dolom": ["Limestone", "Dolomite"],
    "shale, siltstone": ["Shale", "Siltstone"],
    "dolomite, limestone, shale": ["Dolomite", "Limestone", "Shale"],
    "limestone, sandstone, shale": ["Limestone", "Sandstone", "Shale"],
    "dolomite, limestone, sandstone, shale": ["Dolomite", "Limestone", "Sandstone", "Shale"],
    "chert, dolomite, shale": ["Chert", "Dolomite", "Shale"],
    "limestone & dol": ["Limestone", "Dolomite"],
    "anhydrite, sandstone, siltstone": ["Anhydrite", "Sandstone", "Siltstone"],
    "anhydrite, shale": ["Anhydrite", "Shale"],
    "anhydrite, sandstone, shale": ["Anhydrite", "Sandstone", "Shale"],
    "anhydrite, dolomite, limestone, shale": ["Anhydrite", "Dolomite", "Limestone", "Shale"],
    "sandstone, shale, siltstone, other": ["Sandstone", "Shale", "Siltstone"],
    "sand stone": ["Sandstone"],
    "chert, dolomite, sandstone, shale": ["Chert", "Dolomite", "Sandstone", "Shale"],
    "dolomite, sandstone, siltstone": ["Dolomite", "Sandstone", "Siltstone"],
    "chert, dolomite, limestone, sandstone, shale": ["Chert", "Dolomite", "Limestone", "Sandstone", "Shale"],
    "dolomite - lime": ["Dolomite", "Limestone"],
    "ls": ["Limestone"],
    "granitic": ["Granite"],
    "anhydrite and dolomite": ["Anhydrite", "Dolomite"],
    "chert or sandstone": ["Chert", "Sandstone"],
    "dolomite, limestone, sandstone, shale, siltstone": ["Dolomite", "Limestone", "Sandstone", "Shale", "Siltstone"],
    "anhydrite, limestone": ["Anhydrite", "Limestone"],
    "limestone & san": ["Limestone", "Sandstone"],
    "dolomite, siltstone": ["Dolomite", "Siltstone"],
    "lmsh": ["Limestone"],
    "dolomite & lime": ["Dolomite", "Limestone"],
    "limestone - dol": ["Limestone", "Dolomite"],
    "limestone & sha": ["Limestone", "Shale"],
    "chert, limestone, sandstone": ["Chert", "Limestone", "Sandstone"],
    "anhydrite, dolomite, limestone, sandstone, shale": ["Anhydrite", "Dolomite", "Limestone", "Sandstone", "Shale"],
    "limestone, sandstone, shale, siltstone": ["Limestone", "Sandstone", "Shale", "Siltstone"],
    "carbonate, sandstone, shale, siltston": ["Limestone", "Sandstone", "Shale", "Siltstone"],
    "carbonate, sandstone, shale, siltstone": ["Limestone", "Sandstone", "Shale", "Siltstone"],
    "carbonate, sandstone, shale, siltstone, other": ["Limestone", "Sandstone", "Shale", "Siltstone"],
    "anhydrite, dolomite, siltstone": ["Anhydrite", "Dolomite", "Siltstone"],
    "sandstone, siltstone, other": ["Sandstone", "Siltstone"],
    "anhydrite, chert, dolomite, sandstone": ["Anhydrite", "Chert", "Dolomite", "Sandstone"],
    "dolomite, limestone, sandstone, siltstone": ["Dolomite", "Limestone", "Sandstone", "Siltstone"],
    "anhydrite, chert, dolomite, other": ["Anhydrite", "Chert", "Dolomite"],
}

if "temporary_lithology" in df_combined.columns:
    expanded_rows = []
    for _, row in df_combined.iterrows():
        orig = str(row["temporary_lithology"]).strip()
        
        # --------------- leere Werte beibehalten
        if pd.isna(row["temporary_lithology"]) or orig == "" or orig.lower() == "nan":
            new_row = row.copy()
            new_row["rock_type"] = pd.NA
            expanded_rows.append(new_row)
            continue

        # --------------- siehe Excel-Datei | Vereinheitlichung von Trennweisen: Kleinbuchstaben, Trennzeichen, &, ...
        s = orig.lower()
        s = s.replace(";", ",")
        s = s.replace("&", ",")
        s = s.replace(" and ", ",")
        s = s.replace(" or ", ",")
        
        # ---------------- Mehrfache Leerzeichen reduzieren, Kommas vereinheitlichen
        s = re.sub(r"\s+", " ", s)
        s = re.sub(r"\s*,\s*", ", ", s)
        key = s.strip()

        categories = usgs_rock_mapping.get(key)

        # ---------- parts immer definieren
        parts = [p.strip() for p in key.split(",")]
        
        # ---------- Bei Treffer direkt verwenden
        if categories is not None:
            pass  # categories ist bereits korrekt gesetzt
        
        # ---------- Sonderfälle
        else:
            categories = []
            for p in parts:

                if p in {"unknown", "other", "n/a", "na", "none"}:
                    continue

                # ------------------- Für später festgestellte Sonderfälle
                if p in {"ss", "sand stone"}:
                    categories.append("Sandstone")
                elif p == "carbonate":
                    categories.append("Limestone")
                elif p == "sand":
                    categories.append("Sand and Gravel")
                elif p == "san":
                    categories.append("Sandstone")
                elif p == "?":
                    categories.append(np.nan)
                elif p == "sha":
                    categories.append("Shale")  
                elif p == "dol":
                    categories.append("Shale")  
                elif p == "lime":
                    categories.append("Limestone")
                elif p == "ga1n":
                    categories.append(np.nan)
                elif p == "ga1r":
                    categories.append(np.nan)
                elif p == "ga1g":
                    categories.append(np.nan)
                    
                # -------------------- Für Fälle mit Aufspaltung und Zeilenduplizierung
                elif p in {"limestone & dol", "limestone - dol", "limestone dol", "limestone dolom"}:
                    categories.extend(["Limestone", "Dolomite"])
                elif p in {"dolomite & lime", "dolomite - lime"}:
                    categories.extend(["Dolomite", "Limestone"])
                elif p == "limestone & sha":
                    categories.extend(["Limestone", "Shale"])
                elif p == "limestone & san":
                    categories.extend(["Limestone", "Sandstone"])
                elif p == "anhydrite and dolomite":
                    categories.extend(["Anhydrite", "Dolomite"])
                elif p == "chert or sandstone":
                    categories.extend(["Chert", "Sandstone"])
                elif p == "lmsh":
                    categories.append("Limestone")
                elif p == "ls":
                    categories.append("Limestone")
                elif p == "granitic":
                    categories.append("Granite")
                else:
                    categories.append(p.title())
        
        # ---------------- Zeilen duplizieren und rock_type setzen
        for cat in categories:
            new_row = row.copy()
            new_row["rock_type"] = cat
            expanded_rows.append(new_row)

    df_combined = pd.DataFrame(expanded_rows)
    
    # --------------------- Hilfsspalte wird entfernt
    df_combined.drop(columns=["temporary_lithology"], inplace=True)

In [233]:
# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_usgs_path = mid_usgs_path / "before_conversion_usgs_mapped.csv"
import pathlib
_p = pathlib.Path(out_usgs_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_usgs_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_usgs_path.resolve()}")

USGS-Mapping abgeschlossen: 116842 Zeilen, 31 Spalten
CSV-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.4 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [234]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_usgs_path / "before_conversion_usgs_mapped.csv"
excel_path = mid_usgs_path / "before_conversion_usgs_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped.csv
Excel-Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped_readable.xlsx


In [235]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [236]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
import pathlib
_p = pathlib.Path(out_usgs_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_usgs_path, index=False, encoding="utf-8-sig")

In [237]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [238]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number", "Database_name",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan


In [239]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_usgs_path = mid_usgs_path / "after_conversion_usgs_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_usgs_path.parent.mkdir(parents=True, exist_ok=True)
import pathlib
_p = pathlib.Path(after_conversion_usgs_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_usgs_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_usgs_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 116842 Zeilen, 31 Spalten
Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\after_conversion_usgs_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [240]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_usgs_path / "after_conversion_usgs_mapped.csv"
excel_path = mid_usgs_path / "after_conversion_usgs_mapped_readable.xlsx"

In [241]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken" / "Nach_Acquisition"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\after_conversion_usgs_mapped_readable.xlsx


In [242]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
usgs_output_path = output_path / "USGS_cleaned_Database.csv" 
usgs_output_path.parent.mkdir(parents=True, exist_ok=True) 

import pathlib
_p = pathlib.Path(usgs_output_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(usgs_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {usgs_output_path.resolve()}")

Gespeichert unter Pfad: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\USGS_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [243]:
map_path = base_dir / "Kartierung"
usgs_map_path = map_path / "USGS_Map.html" 

In [244]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ------------------------------ Kartenmittelpunkt setzen -----------------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ---------------------- FastMarkerCluster mit Koordinaten füllen ---------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not usgs_map_path.parent.exists(): usgs_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(usgs_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(usgs_map_path)

<div class="alert alert-danger">
    <h1><b>Zusammenfügen der Datenbanken</b></h1>
</div>

In [245]:
from pathlib import Path
import pandas as pd
from datetime import datetime  # falls noch nicht importiert

def merge_csv_folder(output_path) -> Path:

    base = Path(output_path)
    if not base.exists():
        raise FileNotFoundError(f"Ordner existiert nicht: {base.resolve()}")

    csv_files = sorted(p for p in base.glob("*.csv") if p.is_file())
    if not csv_files:
        raise FileNotFoundError(f"Keine .csv-Dateien gefunden in: {base.resolve()}")

    print("Gefundene CSV-Dateien:")
    for p in csv_files:
        print(f"  - {p.name}")

    # -------------------------------- Header der ersten Datei als Referenz ------------------------
    ref_cols = pd.read_csv(csv_files[0], nrows=0).columns.tolist()

    # --------------------------- prüfen, dass alle Header exakt übereinstimmen -------------------
    for p in csv_files[1:]:
        cols = pd.read_csv(p, nrows=0).columns.tolist()
        if cols != ref_cols:
            raise ValueError(
                f"Spaltenkopf unterscheidet sich in Datei: {p.name}\n"
                f"Erwartet: {ref_cols}\n"
                f"Gefunden: {cols}"
            )

    # ----------------------------------------- Einlesen & zusammenfügen --------------------------
    parts = []
    total_rows = 0
    print("\nZeilenanzahl (ohne Header) je Datei:")
    for p in csv_files:
        df_part = pd.read_csv(p, low_memory=False)

        # NEU: Database_name aus Dateinamen ableiten und als Spalte hinzufügen/überschreiben
        db_name = p.stem  # Dateiname ohne .csv
        df_part["Database_name"] = db_name

        parts.append(df_part)
        rows = len(df_part)
        total_rows += rows
        print(f"  - {p.name}: {rows}")

    merged = pd.concat(parts, axis=0, ignore_index=True)

    print(f"\nGesamtzeilen (neu erstellte Datenbank): {len(merged)}")
    assert len(merged) == total_rows, "Summierte Zeilen ≠ Anzahl Zeilen im Merge (unerwartet)."

    # -------------------------------- Zielordner mit Zeitstempel erstellen ----------------------
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  # YYYY-MM-DD-HH-MM-SS
    target_dir = base / "Komplette_Datenbank" / timestamp
    
    target_dir.mkdir(parents=True, exist_ok=True)
    out_path = target_dir / "Komplette_Datenbank.csv"
    import pathlib
    _p = pathlib.Path(out_path)
    if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\nZusammenführung abgeschlossen.")
    print(f"Ausgabedatei: {out_path.resolve()}")

    return out_path, target_dir

# ------------------------------------------- Aufruf ----------------------------------------------
merged_path, target_dir = merge_csv_folder(output_path)
merged = pd.read_csv(merged_path)


Gefundene CSV-Dateien:
  - ALPS_cleaned_Database.csv
  - ARGONNE_cleaned_Database.csv
  - AUSTRALIA-MATARANKA_cleaned_Database.csv
  - CAES2014_cleaned_Database.csv
  - GANDAKI_cleaned_Database.csv
  - HESSEN_cleaned_Database.csv
  - MDPI_Tibet_cleaned_Database.csv
  - REFLECT_cleaned_Database.csv
  - SVZ-CHILE_cleaned_Database.csv
  - THAILAND_cleaned_Database.csv
  - USGS_cleaned_Database.csv
  - XIANSHUIHE_cleaned_Database.csv
  - YUKON_cleaned_Database.csv



Zeilenanzahl (ohne Header) je Datei:
  - ALPS_cleaned_Database.csv: 311


  - ARGONNE_cleaned_Database.csv: 51078
  - AUSTRALIA-MATARANKA_cleaned_Database.csv: 84
  - CAES2014_cleaned_Database.csv: 42
  - GANDAKI_cleaned_Database.csv: 12
  - HESSEN_cleaned_Database.csv: 1144
  - MDPI_Tibet_cleaned_Database.csv: 52
  - REFLECT_cleaned_Database.csv: 5105
  - SVZ-CHILE_cleaned_Database.csv: 30
  - THAILAND_cleaned_Database.csv: 31


  - USGS_cleaned_Database.csv: 116842
  - XIANSHUIHE_cleaned_Database.csv: 280
  - YUKON_cleaned_Database.csv: 88

Gesamtzeilen (neu erstellte Datenbank): 175099



Zusammenführung abgeschlossen.
Ausgabedatei: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\Komplette_Datenbank\2026-03-01_20-40-52\Komplette_Datenbank.csv


C:\Users\lucca\AppData\Local\Temp\ipykernel_20560\966291924.py:71: DtypeWarning: Columns (4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv(merged_path)


In [246]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = merged_path
excel_path = target_dir / "Komplette_Datenbank.xlsx"

In [247]:
import pathlib
_p = pathlib.Path(csv_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\Komplette_Datenbank\2026-03-01_20-40-52\Komplette_Datenbank.xlsx


In [248]:
map_path = base_dir / "Kartierung" / "Fertige_Karte"
merged_map_path = map_path / "Merged_Map.html" 

In [249]:
df = merged.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# Kartenmittelpunkt setzen
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# FastMarkerCluster mit Koordinaten füllen
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

try:
    if not merged_map_path.parent.exists(): merged_map_path.parent.mkdir(parents=True, exist_ok=True)
except:
    pass
import pathlib
_p = pathlib.Path(merged_map_path)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
m.save(merged_map_path)